# Fine-Tune LLaMA 2 7B
This notebook fine-tunes Meta's LLaMA 2 model on a custom review summarization dataset using QLoRA.  Aligned with the JATMO methodology.

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Upload fine-tuning dataset
from google.colab import files
files.upload()  

In [ ]:
# Load and preprocess dataset
import json
from datasets import Dataset

examples = []
with open("llama2_finetune_prompt_response.jsonl", "r") as f:
    for line in f:
        d = json.loads(line)
        prompt = f"### Prompt:\n{d['prompt']}\n\n### Response:\n{d['response']}"
        examples.append({"text": prompt})

dataset = Dataset.from_list(examples)
dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from trl import SFTTrainer
import torch

model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

In [ ]:
training_args = TrainingArguments(
    output_dir="llama2-jatmo-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_total_limit=1,
    fp16=True,
    push_to_hub=False
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    dataset_text_field="text"
)

trainer.train()